> **구분: 수업 후 복습·자습**

# 자습 3. CMake 빌드 시스템 — 약 60분

여러 C++ 파일을 매번 긴 `g++` 명령으로 조합하는 대신, **무엇을 만들고 서로 어떻게 연결할지** target으로 선언하는 방법을 배웁니다. 통계 라이브러리, 실행 파일, 테스트를 가진 작은 프로젝트를 구성·빌드·검사합니다.

> 실행 환경: JupyterLab의 Python 3 커널 + CMake 3.16 이상 + C++17 컴파일러  
> 구현 폴더: `06_cmake/{exercise,solution}/`  
> exercise source는 아래 `%%writefile` 셀이 저장하며 빌드 결과는 `/workspace/Data/build/03.C++/06_cmake`에 만듭니다.

<!-- standalone-review:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** Python 3 커널에서 첫 환경 셀의 `/opt/course/bin/python` 경로를 확인합니다. 코드 셀은 Shift+Enter로 실행하고 `[*]`가 끝난 뒤 다음 셀로 이동합니다. TODO 파일을 저장한 뒤 검증 셀을 실행합니다.

### 이 노트북만으로 시작하기

열 파일은 `/workspace/03.C++/92_selfstudy_cmake.ipynb`이며 선택 자습입니다. [03_memory_stl_cmake.ipynb의 「CMake는 빌드 대상을 연결합니다」](03_memory_stl_cmake.ipynb)에 기초가 있습니다. 필요한 내용은 소스 `.cpp`를 컴파일하고 링크해 앱·라이브러리를 만든다는 점이며, `cmake -S 소스 -B 빌드`가 configure, `cmake --build 빌드`가 build, `ctest --test-dir 빌드 --output-on-failure`가 test입니다. 여기서는 이 명령을 현재 노트북의 도우미로 다시 실행합니다.

이전 빌드·변수는 사용하지 않습니다. 첫 환경 셀과 바로 뒤 `project_paths`, `configure`, `build` 도우미 셀을 실행합니다. 세 `%%writefile` 셀이 `/workspace/03.C++/06_cmake/exercise/`의 `CMakeLists.txt`, `src/statistics.cpp`, `tests/statistics_tests.cpp`를 저장합니다. 외부 파일에 최신 답이 있으면 작성 셀에 먼저 옮기세요. 산출물은 `/workspace/Data/build/03.C++/06_cmake/`에 저장됩니다.
<!-- standalone-review:end -->


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

여러 소스와 라이브러리의 관계를 사람이 매번 긴 컴파일 명령으로 맞추는 대신 CMake의 target으로 표현합니다. 설정·빌드·테스트가 각각 무엇을 만드는지 이해하면 결과가 갱신되지 않거나 테스트가 실행되지 않는 문제를 구분할 수 있습니다.

### 전체 과정에서 내 코드가 하는 일

```text
파일·target 관계 읽기
  ↓
CMake TODO
  ↓
configure
  ↓
build
  ↓
테스트 발견·실행
  ↓
소스 변경 후 재빌드
```

실행 파일과 라이브러리의 의존 관계, 테스트 등록을 완성합니다. 처음 제공된 일부 테스트의 성공을 과제 전체 완료로 보지 말고 뒤의 독립 계약 검사를 수행합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** CTest가 오류 없이 끝났는데 테스트가 0개라면?

**설명:** 검사가 실행되지 않았으므로 통과 증거가 아닙니다. 테스트 활성화·등록과 빌드 디렉터리를 확인합니다. configure 성공은 설정 생성이고 build 성공은 산출물 생성이며 CTest의 실제 검사 성공이 다음 단계입니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| `undefined reference`, 헤더 없음 | 선언/정의와 CMake의 소스 목록·include 경로·target 연결을 점검합니다. | configure와 build 모두 성공 |
| `No tests were found`, 검사 일부만 실행 | 올바른 build 폴더인지, enable_testing/add_test와 full_contract_enabled 설정을 확인하고 재configure/build합니다. | statistics.unit와 전체 계약 완료 메시지 확인 |
| CMakeCache 경로 불일치 | 다른 경로/컴파일러의 cache 재사용인지 확인하고 기존 결과와 분리한 새 build 폴더에서 configure합니다. | 새 폴더의 build/test 성공 |
| 수정한 결과가 반영되지 않음 | 본인 exercise 소스를 저장하고 빌드 셀부터 다시 실행합니다. 이전 실행 파일을 실행했는지 확인합니다. | 새 빌드의 학습자 검사 PASS |

**예시에 없는 오류를 확인할 곳:** [C++ 공통 안내서](BEGINNER_GUIDE.md), 해당 lesson의 tests와 빌드 stderr. CMake는 build 폴더의 `Testing/Temporary/LastTest.log`도 확인합니다. [CTest 공식 안내](https://cmake.org/cmake/help/latest/manual/ctest.1.html), [sanitizer 옵션](https://gcc.gnu.org/onlinedocs/gcc/Instrumentation-Options.html).


[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 후 복습·자습**

이 노트북은 수업 후 자습입니다. 본수업 필수 완료 조건에 포함하지 않습니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 제공된 빌드 틀에서 지정한 TODO를 수정하고 검증합니다.
- **단원 마지막 연결 질문:** Python 결과와 같음을 먼저 확인합니다. 기존 C++ 부품 재사용, 호출 비용, 수정·배포 부담 중 이 기능을 .so로 분리할 이유를 하나 선택합니다. C++이라는 이유만으로 빨라진다고 결론내리지 않습니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.


> **구분: 수업 후 복습·자습**

## 처음 학습하는 학생을 위한 노트북 사용법

이 노트북의 설명은 위에서 아래로 읽되, 모든 문법을 외운 뒤 코드를 실행하려고 하지 않아도 됩니다. **코드 읽기 → 결과 예상 → 셀 실행 → 실제 결과 비교 → TODO 수정 → 검증** 순서로 반복하면 됩니다.

| 셀 또는 파일 | 하는 일 | 학생이 할 일 |
|---|---|---|
| Markdown 설명 | 새 용어와 코드의 목적 설명 | 낯선 용어를 표시하고 실행 결과를 먼저 예상 |
| 환경 Python 셀 | `g++`·CMake를 호출할 도우미와 경로 준비 | 내용을 바꾸지 말고 먼저 실행 |
| `examples/` | 완성된 작은 C++ 예제 | 출력 전에 변수 변화를 손으로 추적 |
| `exercise/` | 학생이 완성할 TODO | 지정된 부분만 수정하고 파일 저장 |
| build 셀 | 소스를 컴파일하고 링크 | 첫 오류부터 읽고 수정 후 다시 실행 |
| learner verification | 자신의 코드가 계약을 만족하는지 검사 | 정답 확인 시점은 강사가 조절 |
| 접힌 solution | 한 가지 참고 구현 | 통과 후 자신의 코드와 다른 이유를 비교 |

이 노트북의 커널은 Python입니다. Python 셀이 C++ 파일을 `g++` 또는 CMake에 전달하고 결과를 화면에 보여 줍니다. 마지막의 `CalledProcessError`만 읽지 말고, 그 위에 나온 컴파일러 또는 CTest의 **첫 번째 오류** 를 먼저 찾으세요. 공통 용어와 문제 해결 절차는 `03.C++/BEGINNER_GUIDE.md`에 더 자세히 정리되어 있습니다.

> **구분: 수업 후 복습·자습**

## 이번 과정에서의 위치

**자습 확장 / 약 60분, 본수업 4시간에 포함하지 않음** . Python 커널로 실제 g++/CMake를 실행합니다. 먼저 환경 셀을 실행하세요.
원격 사용자도 GUI 없이 동일하게 진행합니다. 소스 저장 셀은 명시한 exercise 파일을 덮어쓰므로 노트북과 외부 편집기에서 동시에 수정하지 마세요.
미완성 TODO는 빌드에 성공해도 학습자 검증에 실패할 수 있습니다. 자신의 검증→접힌 정답→정답 검증 순서입니다.
빌드 결과는 `/workspace/Data/build/03.C++`에 보존됩니다. 컨테이너 재생성 후에는 환경 셀부터 실행합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
from pathlib import Path
import os
import shlex
import subprocess
import tempfile

def locate_course_dir():
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        if base.name == '03.C++':
            return base
        candidate = base / '03.C++'
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError('03.C++ 폴더를 찾지 못했습니다. workspace 또는 03.C++에서 노트북을 여세요.')

def run(command, *, cwd=None):
    args = [str(part) for part in command]
    print('$', shlex.join(args))
    completed = subprocess.run(
        args, cwd=str(cwd or COURSE_DIR), text=True, capture_output=True
    )
    if completed.stdout:
        print(completed.stdout, end='')
    if completed.stderr:
        print(completed.stderr, end='')
    completed.check_returncode()
    return completed

COURSE_DIR = locate_course_dir()
LESSON_DIR = COURSE_DIR / '06_cmake'
BUILD_ROOT = COURSE_DIR.parent / 'Data' / 'build' / '03.C++' / LESSON_DIR.name
os.chdir(COURSE_DIR.parent)
print('과정 폴더:', COURSE_DIR)
print('이번 구현 폴더:', LESSON_DIR)
print('호스트에 보존되는 빌드 루트:', BUILD_ROOT)
import sys
print('Python:', sys.executable)


> **구분: 수업 후 복습·자습**

## 오늘의 60분 지도

| 시간 | 주제 | 도착점 |
|---:|---|---|
| 0–6분 | 환경·구조 확인 | 프로젝트의 파일 역할 구분 |
| 6–15분 | 구성과 빌드 | `-S`, `-B`, `--build` 설명 |
| 15–28분 | target | 라이브러리와 실행 파일 연결 |
| 28–38분 | 사용 요구사항 | include 경로와 `PUBLIC`/`PRIVATE` 이해 |
| 38–45분 | CTest | 테스트 target 등록·실행 |
| 45–56분 | TODO 실습 | 통계 구현 후 앱과 테스트 검증 |
| 56–60분 | 해답·퀴즈 | 빌드 그래프를 말로 설명 |

> **구분: 수업 후 복습·자습**

## 학습 목표와 실습 방법

이 노트북을 마치면 다음을 할 수 있습니다.

1. CMake의 구성(configure)과 빌드(build)를 구분한다.
2. 정적 라이브러리, 실행 파일, 테스트를 각각 target으로 선언한다.
3. `target_link_libraries`가 만드는 의존 관계를 설명한다.
4. `target_include_directories`의 `PUBLIC`과 `PRIVATE`를 구분한다.
5. CTest로 등록된 테스트를 실행하고 실패 출력을 읽는다.

TODO 시간에는 아래 세 `%%writefile` 셀에서 CMake와 C++ source를 편집·저장한 뒤 build와 run/test 셀을 차례로 실행하세요.

In [ ]:
# 구분: 수업 후 복습·자습 셀
assert LESSON_DIR.is_dir(), f'구현 폴더가 없습니다: {LESSON_DIR}'
run(['g++', '--version'])
run(['cmake', '--version'])
run(['ctest', '--version'])
print('프로젝트 파일:')
for path in sorted(LESSON_DIR.rglob('*')):
    if path.is_file():
        print(' -', path.relative_to(LESSON_DIR))

In [ ]:
# 구분: 수업 후 복습·자습 셀
def project_paths(variant):
    source_dir = LESSON_DIR / variant
    build_dir = BUILD_ROOT / variant
    if not (source_dir / 'CMakeLists.txt').is_file():
        raise FileNotFoundError(source_dir / 'CMakeLists.txt')
    return source_dir, build_dir

def configure(variant):
    source_dir, build_dir = project_paths(variant)
    run([
        'cmake', '-S', source_dir, '-B', build_dir,
        '-DCMAKE_BUILD_TYPE=Debug', '-DBUILD_TESTING=ON',
    ])
    return build_dir

def build(variant, *targets):
    build_dir = configure(variant)
    command = ['cmake', '--build', build_dir, '--parallel', '2']
    if targets:
        command.extend(['--target', *targets])
    run(command)
    return build_dir

print('CMake 프로젝트 도우미 준비 완료')

> **구분: 수업 후 복습·자습**

## CMake를 처음 보는 학생의 핵심 구분

`g++`는 C++ 소스를 실제로 번역하는 컴파일러이고, CMake는 어떤 파일과 옵션으로 어떤 결과물을 만들지를 기술한 프로젝트를 읽어 빌드 규칙을 생성합니다. CTest는 등록된 테스트 실행 파일을 실행하고 성공 여부를 모읍니다.

```text
CMakeLists.txt
    ↓ cmake configure
build 디렉터리의 빌드 규칙
    ↓ cmake --build가 g++ 등을 호출
목적 파일 + 라이브러리 + 실행 파일
    ↓ ctest
등록된 테스트 실행과 결과 요약
```

configure가 성공했다고 C++ 소스가 모두 컴파일된 것은 아닙니다. build가 성공했다고 프로그램의 계산이 맞는 것도 아닙니다. test가 성공해야 현재 등록된 입력 계약을 만족했다고 말할 수 있습니다.

소스 디렉터리는 사람이 관리할 파일이고 빌드 디렉터리는 다시 만들 수 있는 생성물입니다. 빌드가 이상할 때 빌드 디렉터리를 새로 만들 수 있어도, exercise 소스를 지우면 자신의 작업을 잃습니다. 두 경로를 항상 구분하세요.

> **구분: 수업 후 복습·자습**

---
## 1. 소스 트리와 빌드 트리를 분리한다 (6–15분)

<!-- procedure:lesson-09-d131f2:start -->
![소스·target·테스트 연결하기](../교재그림/절차/lesson-09.svg)

실행 파일이 만들어져도 테스트가 등록되지 않으면 CTest가 해당 동작을 검사하지 않습니다.
<!-- procedure:lesson-09-d131f2:end -->


이 프로젝트는 역할별로 파일을 나눕니다.

```text
exercise/
├── CMakeLists.txt
├── include/statistics/statistics.hpp  # 공개 선언
├── src/statistics.cpp                 # 라이브러리 구현
├── src/main.cpp                       # 기본 앱 진입점
├── src/report.cpp                     # 직접 target으로 추가할 보고서 앱
└── tests/statistics_tests.cpp         # 테스트 진입점
```

`cmake -S <source> -B <build>`는 `CMakeLists.txt`를 읽어 빌드 시스템을 **구성** 합니다. `cmake --build <build>`는 생성된 빌드 시스템으로 실제 컴파일·링크를 수행합니다. 빌드 산출물을 별도 폴더에 두면 소스 트리가 깨끗하고 여러 설정을 따로 유지하기 쉽습니다.

> **구분: 수업 후 복습·자습**

### 실행 전 예측 — 아직 TODO 상태인 앱

현재 `exercise/src/statistics.cpp`에서 `mean()`은 `0.0`, `median()`은 첫 원소를 임시 반환합니다. `{8, 3, 5, 4}`를 입력하면 다음 중 무엇이 출력될지 예상하세요.

- 개수와 최솟값/최댓값은 이미 올바를까요?
- 평균은 `0.00`과 `5.00` 중 무엇일까요?
- 중앙값은 `8.00`과 `4.50` 중 무엇일까요?

아래 셀은 먼저 configure하고 `statistics_app` target만 요청합니다. 앱이 링크하는 `statistics` 라이브러리도 함께 빌드되는지 로그에서 찾으세요.

> **구분: 수업 후 복습·자습**

### 노트북 편집 → 파일 저장 계약

아래 펼친 `%%writefile` 셀이 [03.C++/06_cmake/exercise/CMakeLists.txt](06_cmake/exercise/CMakeLists.txt), [03.C++/06_cmake/exercise/src/statistics.cpp](06_cmake/exercise/src/statistics.cpp), [03.C++/06_cmake/exercise/tests/statistics_tests.cpp](06_cmake/exercise/tests/statistics_tests.cpp)의 학습 원본입니다. 셀의 `TODO`를 수정한 뒤 실행하면 기존 파일을 덮어씁니다. 그 다음 build/run 셀을 실행하세요. 외부 편집기와 동시에 수정하지 말고, 노트북을 저장해 수정한 셀을 보존하세요.


> **구분: 수업 후 복습·자습 셀**

### 바로 아래 CMake TODO — 보고서 앱 target 추가

작성 파일은 `03.C++/06_cmake/exercise/CMakeLists.txt`입니다. `src/report.cpp`는 제공된 보고서 앱 소스이며 `statistics`는 평균·중앙값을 제공하는 정적 라이브러리입니다. TODO의 세 줄을 활성화해 (1) `statistics_report` 실행 target 생성, (2) `statistics`를 PRIVATE으로 연결, (3) `enable_strict_warnings(statistics_report)` 적용을 모두 수행하세요. 기존 앱·테스트 target은 유지합니다. 새 target 선언 뒤에는 configure를 다시 실행해야 합니다. 전체 설명은 아래 「실습 목적과 TODO 구현 명세」에 있으며 여기서 완료 결과는 보고서 실행 파일 생성과 `보고서 평균/중앙값: 5.00 / 5.00` 출력입니다. 통계 함수 TODO도 완성해야 그 값이 맞습니다.


In [ ]:
%%writefile 03.C++/06_cmake/exercise/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

project(statistics_exercise LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

function(enable_strict_warnings target_name)
    if(MSVC)
        target_compile_options(${target_name} PRIVATE /W4 /WX /permissive-)
    else()
        target_compile_options(
            ${target_name}
            PRIVATE
                -Wall
                -Wextra
                -Wpedantic
                -Wconversion
                -Wshadow
                -Werror
        )
    endif()
endfunction()

add_library(statistics STATIC src/statistics.cpp)
target_include_directories(
    statistics
    PUBLIC
        ${CMAKE_CURRENT_SOURCE_DIR}/include
)
enable_strict_warnings(statistics)

add_executable(statistics_app src/main.cpp)
target_link_libraries(statistics_app PRIVATE statistics)
enable_strict_warnings(statistics_app)

# TODO(CMake): 아래 세 줄의 주석을 풀어 두 번째 실행 target을 추가하세요.
# add_executable(statistics_report src/report.cpp)
# target_link_libraries(statistics_report PRIVATE statistics)
# enable_strict_warnings(statistics_report)

include(CTest)
if(BUILD_TESTING)
    add_executable(statistics_tests tests/statistics_tests.cpp)
    target_link_libraries(statistics_tests PRIVATE statistics)
    enable_strict_warnings(statistics_tests)
    add_test(NAME statistics.unit COMMAND statistics_tests)
endif()


> **구분: 수업 후 복습·자습 셀**

### 바로 아래 C++ TODO — 평균과 중앙값

작성 파일은 `03.C++/06_cmake/exercise/src/statistics.cpp`입니다. 입력은 실수 `vector<double>`이며 반환은 double 한 값입니다. **mean:** `<numeric>`를 추가하고 `std::accumulate`의 시작값을 `0.0`으로 두어 실수 합계를 구한 뒤 원소 수로 나눕니다. 읽기 전용 참조이므로 원본은 바꾸지 않습니다. **median:** 매개변수는 값으로 받은 복사본이므로 이를 정렬하고, 길이 n이 홀수이면 인덱스 `n/2`, 짝수이면 `n/2-1`과 `n/2`의 두 값 평균을 반환합니다. 호출자의 순서는 유지해야 합니다.

`require_values`의 빈 입력 `std::invalid_argument` 예외는 그대로 유지합니다. `{8,3,5,4}`는 평균 5.0·중앙값 4.5, `{9,1,5}`는 중앙값 5.0, `{-9,-1,-5}`는 평균·중앙값 -5.0이어야 합니다. `summarize`의 제공 count/min/max 계산은 바꾸지 않습니다. 자세한 위치는 아래 「실습 목적과 TODO 구현 명세」이며 여기의 계약으로도 두 함수를 작성할 수 있습니다.


In [ ]:
%%writefile 03.C++/06_cmake/exercise/src/statistics.cpp
#include "statistics/statistics.hpp"

#include <algorithm>
#include <stdexcept>

namespace training::statistics {
namespace {

void require_values(const std::vector<double>& values) {
    if (values.empty()) {
        throw std::invalid_argument("통계를 계산할 값이 필요합니다.");
    }
}

}  // namespace

double mean(const std::vector<double>& values) {
    require_values(values);

    // TODO: <numeric>의 std::accumulate로 합계를 구하고 개수로 나누세요.
    return 0.0;
}

double median(std::vector<double> values) {
    require_values(values);

    // TODO: 값을 정렬하고 홀수/짝수 개수에 맞는 중앙값을 반환하세요.
    return values.front();
}

Summary summarize(const std::vector<double>& values) {
    require_values(values);
    const auto bounds = std::minmax_element(values.begin(), values.end());

    return Summary{
        values.size(),
        *bounds.first,
        *bounds.second,
        mean(values),
        median(values),
    };
}

}  // namespace training::statistics


> **구분: 수업 후 복습·자습 셀**

### 바로 아래 테스트 TODO — 전체 경계 검사 활성화

작성 파일은 `03.C++/06_cmake/exercise/tests/statistics_tests.cpp`입니다. `full_contract_enabled`의 `false`를 `true`로 바꾸면 지금까지 생략된 평균·홀짝 중앙값·음수·빈 입력 예외·원본 순서 보존 검사가 함께 실행됩니다. expected 값이나 검사 조건은 바꾸지 않습니다. 실행 입력과 기대값은 아래 테스트 코드에 제공되어 있으며 실패하면 앞 통계 구현으로 돌아가 수정·저장·재빌드합니다. 기본 CTest 통과 메시지는 아직 전체 검사가 꺼졌다는 뜻입니다. `모든 통계 계약을 통과했습니다.` 및 아래 최종 학습자 검증 통과까지 확인해야 완료입니다.


In [ ]:
%%writefile 03.C++/06_cmake/exercise/tests/statistics_tests.cpp
#include "statistics/statistics.hpp"

#include <cmath>
#include <iostream>
#include <stdexcept>
#include <string>
#include <vector>

namespace {

void expect(bool condition, const std::string& message) {
    if (!condition) {
        throw std::runtime_error(message);
    }
}

void expect_close(double actual, double expected, const std::string& label) {
    constexpr double tolerance = 1.0e-12;
    expect(std::abs(actual - expected) < tolerance, label);
}

}  // namespace

int main() {
    using training::statistics::mean;
    using training::statistics::median;
    using training::statistics::summarize;

    const std::vector<double> samples{8.0, 3.0, 5.0, 4.0};
    const std::vector<double> samples_before{samples};
    const auto summary = summarize(samples);

    expect(summary.count == 4U, "대표 입력: count");
    expect_close(summary.minimum, 3.0, "대표 입력: minimum");
    expect_close(summary.maximum, 8.0, "대표 입력: maximum");

    // TODO(test): 아래 값을 false에서 true로 바꾸면 모든 경계 계약이 함께 실행됩니다.
    constexpr bool full_contract_enabled{false};
    if (!full_contract_enabled) {
        std::cout << "기본 CTest 통과: full_contract_enabled를 true로 바꾸세요.\n";
        return 0;
    }

    expect_close(summary.mean, 5.0, "대표 입력: mean");
    expect_close(summary.median, 4.5, "짝수 입력: median");
    expect(samples == samples_before, "summarize는 원본 순서를 보존해야 합니다.");

    const std::vector<double> odd_values{9.0, 1.0, 5.0};
    const std::vector<double> odd_before{odd_values};
    expect_close(median(odd_values), 5.0, "홀수 입력: median");
    expect(odd_values == odd_before, "median은 호출자의 원본을 바꾸면 안 됩니다.");

    const std::vector<double> negative_values{-9.0, -1.0, -5.0};
    expect_close(mean(negative_values), -5.0, "음수 입력: mean");
    expect_close(median(negative_values), -5.0, "음수 입력: median");

    bool mean_rejected_empty = false;
    try {
        static_cast<void>(mean({}));
    } catch (const std::invalid_argument&) {
        mean_rejected_empty = true;
    }
    expect(mean_rejected_empty, "mean은 빈 입력을 거부해야 합니다.");

    bool median_rejected_empty = false;
    try {
        static_cast<void>(median({}));
    } catch (const std::invalid_argument&) {
        median_rejected_empty = true;
    }
    expect(median_rejected_empty, "median은 빈 입력을 거부해야 합니다.");

    bool summarize_rejected_empty = false;
    try {
        static_cast<void>(summarize({}));
    } catch (const std::invalid_argument&) {
        summarize_rejected_empty = true;
    }
    expect(summarize_rejected_empty, "summarize는 빈 입력을 거부해야 합니다.");

    std::cout << "모든 통계 계약을 통과했습니다.\n";
}


In [ ]:
# 구분: 수업 후 복습·자습 셀
from hashlib import sha256
from pathlib import Path

active_sources = [
    '03.C++/06_cmake/exercise/CMakeLists.txt',
    '03.C++/06_cmake/exercise/src/statistics.cpp',
    '03.C++/06_cmake/exercise/tests/statistics_tests.cpp'
]
for relative_path in active_sources:
    source_path = Path(relative_path)
    payload = source_path.read_bytes()
    print(f'{source_path}: sha256={sha256(payload).hexdigest()}')


In [ ]:
# 구분: 수업 후 복습·자습 셀
exercise_build = build('exercise', 'statistics_app')


In [ ]:
# 구분: 수업 후 복습·자습 셀
run([exercise_build / 'statistics_app'])

> **구분: 수업 후 복습·자습**

## 2. target으로 빌드 그래프 표현하기 (15–28분)

핵심 선언은 파일 나열보다 **target 간 관계** 입니다.

```cmake
add_library(statistics STATIC src/statistics.cpp)
add_executable(statistics_app src/main.cpp)
target_link_libraries(statistics_app PRIVATE statistics)
```

- `statistics`: 재사용할 정적 라이브러리 target
- `statistics_app`: `main()`을 가진 실행 파일 target
- 링크 선언: 앱이 라이브러리보다 뒤에 빌드되어야 하며 그 결과를 링크한다는 의존 관계

target 이름은 CMake 안에서 쓰는 논리적 이름입니다. 소스 파일명과 같을 필요가 없습니다. 한 target에 관련 설정을 모으면 새 실행 파일이나 테스트가 추가되어도 전역 플래그에 덜 의존합니다.

In [ ]:
# 구분: 수업 후 복습·자습 셀
cmake_text = (LESSON_DIR / 'exercise' / 'CMakeLists.txt').read_text(encoding='utf-8')
active_cmake = '\n'.join(
    line for line in cmake_text.splitlines()
    if line.strip() and not line.lstrip().startswith('#')
)
for keyword in ('add_library', 'add_executable', 'target_link_libraries', 'add_test'):
    print(f'{keyword:27} ->', active_cmake.count(keyword), '회 (활성 선언)')

> **구분: 수업 후 복습·자습**

## 3. 사용 요구사항과 `PUBLIC`/`PRIVATE` (28–38분)

라이브러리 공개 헤더는 `include/statistics/statistics.hpp`에 있습니다. `statistics.cpp`뿐 아니라 이를 링크하는 앱과 테스트도 그 경로를 알아야 하므로 include 경로는 `PUBLIC`입니다.

```cmake
target_include_directories(statistics
    PUBLIC ${CMAKE_CURRENT_SOURCE_DIR}/include
)
```

- `PRIVATE`: 이 target을 만들 때만 필요
- `INTERFACE`: 이 target을 사용하는 소비자에게만 필요
- `PUBLIC`: 자신과 소비자 모두 필요

### 전이되는 usage requirement

CMake에서 `target_link_libraries(app PRIVATE statistics)`는 라이브러리 파일만 뒤에 붙이는 명령이 아니라, `statistics`가 공개한 **사용 요구사항(usage requirements)** 을 소비자 `app`에 전달하는 그래프 간선입니다. `statistics`의 `PUBLIC`/`INTERFACE` include 경로·컴파일 정의·컴파일 기능은 직접 소비자에게 전이되지만 `PRIVATE` 항목은 멈춥니다. 그래서 공개 헤더가 요구하는 설정은 인터페이스에, 구현 `.cpp`만 요구하는 경고나 내부 정의는 `PRIVATE`에 두어야 합니다. 전역 `include_directories()`보다 target 기반 선언이 의존 관계와 전파 범위를 정확히 표현하는 이유입니다.

### 30초 예측

include 경로를 `PRIVATE`로 바꾸면 `statistics.cpp`는 컴파일될 수 있어도, `statistics_app`의 `#include "statistics/statistics.hpp"`는 성공할까요? 링크 관계가 include 경로도 전달해야 하는 이유를 말해 보세요.

> **구분: 수업 후 복습·자습**

### 표준과 경고도 target 가까이에 두기

이 프로젝트는 C++17을 필수로 하고 컴파일러 확장을 끕니다. 또한 각 target에 엄격한 경고를 적용합니다.

```cmake
set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)
target_compile_options(statistics PRIVATE -Wall -Wextra -Wpedantic)
```

더 큰 프로젝트에서는 `target_compile_features(target PUBLIC cxx_std_17)`처럼 요구사항을 target 단위로 전파할 수도 있습니다. 여기서는 모든 target이 같은 표준을 쓰므로 전역 표준 설정을 사용합니다.

> **구분: 수업 후 복습·자습**

### target 중심으로 오류 범위를 좁히기

각 target은 자신의 소스, 헤더 검색 경로, 컴파일 옵션, 링크 의존성을 가집니다. 오류 메시지에서 어떤 target을 만드는 중이었는지 보면 조사 범위가 줄어듭니다.

| 실패 위치 | 흔한 원인 | 먼저 확인할 선언 |
|---|---|---|
| configure | CMake 문법, 존재하지 않는 source 파일 | `add_library`, `add_executable`의 경로 |
| library compile | 구현 문법·타입, 헤더 검색 | `target_include_directories`, 구현 파일 |
| executable compile | main/report/test 자체의 오류 | 해당 target의 source와 include |
| link | 선언과 정의 불일치, 라이브러리 연결 누락 | `target_link_libraries`, 함수 서명 |
| CTest | 프로그램은 실행되지만 계약 불만족 | 실패 테스트 출력과 구현 로직 |

`PUBLIC`과 `PRIVATE`는 누가 헤더를 읽을 수 있는지에 대한 일반 접근 제어자가 아니라, target의 사용 요구사항이 소비자에게 전파되는지를 정합니다. 라이브러리 공개 헤더가 특정 include 경로를 요구하고 소비자도 그 헤더를 포함해야 한다면 그 경로는 보통 `PUBLIC`입니다. 구현 파일만 필요한 경로라면 `PRIVATE`입니다.

> **구분: 수업 후 복습·자습**

## 4. CTest: 실행 명령을 테스트로 등록하기 (38–45분)

`include(CTest)`는 `BUILD_TESTING` 옵션과 테스트 기능을 준비합니다. 테스트 실행 파일도 일반 target이며, `add_test`가 CTest에 실행 방법을 등록합니다.

```cmake
include(CTest)
if(BUILD_TESTING)
    add_executable(statistics_tests tests/statistics_tests.cpp)
    target_link_libraries(statistics_tests PRIVATE statistics)
    add_test(NAME statistics.unit COMMAND statistics_tests)
endif()
```

`cmake --build`는 테스트 프로그램을 **만들고** , 빌드 디렉터리에서 실행한 `ctest --output-on-failure`는 등록된 테스트를 **실행** 합니다. 두 단계는 다릅니다. 이 방식은 CMake 3.16에서도 동작하며, 노트북은 Python의 `cwd=`로 CTest 작업 디렉터리를 지정합니다. 현재 starter 테스트는 `full_contract_enabled`가 `false`라 기본 검사만 수행합니다. 값을 `true`로 바꾸면 모든 경계 계약이 함께 실행됩니다.

In [ ]:
# 구분: 수업 후 복습·자습 셀
exercise_build = build('exercise', 'statistics_tests')


In [ ]:
# 구분: 수업 후 복습·자습 셀
run(['ctest', '--output-on-failure'], cwd=exercise_build)

> **구분: 수업 후 복습·자습**

### 흔한 실수

- `CMakeLists.txt`를 바꿨지만 build만 반복하고 configure가 다시 필요하다는 점을 놓친다. `cmake --build`가 자동 재구성하는 경우도 있지만 명시적으로 다시 configure하면 원인을 분리하기 쉽습니다.
- 소스 폴더 안에 생성 파일을 섞는다. `-S`와 `-B`로 out-of-source build를 사용하세요.
- 전역 `include_directories()`와 `link_libraries()`에 모든 설정을 모은다. 소비 관계를 target별로 선언하세요.
- 헤더를 사용하는 소비자에게 필요한 경로를 `PRIVATE`로 둔다. 전파 범위를 생각하세요.
- 테스트 실행 파일만 만들고 `add_test`를 빠뜨린다. 이 경우 CTest는 테스트를 찾지 못합니다.
- 이전 캐시 때문에 이상한 결과가 날 때 소스까지 지운다. 빌드 디렉터리만 새 경로로 바꾸면 소스는 안전합니다.

> **구분: 수업 후 복습·자습**

---
## 5. TODO 실습 — target 추가와 통계 라이브러리 완성 (45–56분)

위의 세 `%%writefile` 셀에서 다음 파일을 직접 수정하고 각각 실행해 저장하세요.

1. [03.C++/06_cmake/exercise/CMakeLists.txt](06_cmake/exercise/CMakeLists.txt)
   - `TODO(CMake)` 아래 세 줄의 주석을 풀어 `statistics_report` 실행 target 선언
   - `src/report.cpp`를 소스로 쓰고 `statistics` 라이브러리를 `PRIVATE`로 링크
2. [03.C++/06_cmake/exercise/src/statistics.cpp](06_cmake/exercise/src/statistics.cpp)
   - `<numeric>` 추가
   - `std::accumulate`로 평균 구현
   - 복사된 값을 정렬하고 홀수/짝수 중앙값 구현
3. [03.C++/06_cmake/exercise/tests/statistics_tests.cpp](06_cmake/exercise/tests/statistics_tests.cpp)
   - `full_contract_enabled`를 `false`에서 `true`로 바꾸어 전체 계약 검사 활성화

즉, 이번에는 완성된 CMake를 읽기만 하지 않고 두 번째 실행 target을 직접 빌드 그래프에 추가합니다. 저장 뒤 configure → build → run → CTest 흐름을 다시 수행하세요.

> **구분: 수업 후 복습·자습**

### starter 코드 해설 1 — 파일 역할과 target 의존 그래프

- `include/statistics/statistics.hpp`는 소비자에게 공개할 계약입니다. `Summary`의 다섯 필드와 `mean`, `median`, `summarize`의 입력·반환 형식만 선언합니다.
- `src/statistics.cpp`는 그 계약의 구현입니다. `require_values`가 빈 벡터를 예외로 막고, `summarize`는 이미 개수·최솟값·최댓값을 구한 뒤 TODO 함수들을 호출합니다.
- `src/main.cpp`는 `{8, 3, 5, 4}`의 전체 요약을 출력하는 기본 앱입니다. 라이브러리의 사용법을 보여 주지만 라이브러리 구현을 포함하지는 않습니다.
- `src/report.cpp`는 `{2, 4, 6, 8}`의 평균과 중앙값만 출력하는 두 번째 앱의 완성된 진입점입니다. 파일이 존재하는 것만으로는 CMake target이 되지 않습니다.
- `tests/statistics_tests.cpp`는 작은 `expect`/`expect_close` 도우미로 요약 결과를 확인하는 테스트 진입점입니다. `expect_close`는 부동소수점 값을 완전 일치 대신 허용 오차로 비교합니다.
- `CMakeLists.txt`는 위 파일을 어떤 target으로 묶고 연결할지 선언하며, C++17과 엄격한 경고를 각 target에 적용합니다.

```text
statistics (STATIC: src/statistics.cpp, PUBLIC include 경로)
├──▶ statistics_app    (src/main.cpp)
├──▶ statistics_report (src/report.cpp, TODO에서 활성화)
└──▶ statistics_tests  (tests/statistics_tests.cpp, BUILD_TESTING=ON)
                         └── CTest 이름 statistics.unit로 실행 등록
```

화살표는 각 실행 target이 `statistics`를 링크하므로 라이브러리가 먼저 빌드되고 공개 include 경로도 전달된다는 뜻입니다. `add_test`는 새 컴파일 의존성을 만드는 명령이 아니라, 이미 만들어진 `statistics_tests`를 CTest가 어떤 이름과 명령으로 실행할지 등록합니다. starter에서 `statistics_report` 세 줄이 주석인 이유는 `report.cpp` 작성이 아니라 **기존 소스를 새 target으로 빌드 그래프에 편입하는 연습** 을 남겨 두기 위해서입니다. 그래서 초기 configure/build는 report 없이도 성공합니다.

> **구분: 수업 후 복습·자습**

### 실습 목적과 TODO 구현 명세

- **실습 목적:** 통계 라이브러리의 계산 구현과 소비 target·CTest를 하나의 CMake 의존 그래프로 연결합니다.
- **구현 항목:** report target, `mean`, `median`, 전체 계약 검사 스위치를 활성화하고 configure부터 test까지 수행합니다.
- **입력/출력:** `vector<double>` 샘플을 입력으로 요약 통계와 report 평균·중앙값을 출력하며 CTest가 함수 결과를 판정합니다.
- **제약:** 빈 입력의 기존 예외 계약과 호출자 벡터 순서를 유지하고, target 기반 include/link 관계 및 C++17 설정을 깨뜨리지 않습니다.
- **완료 기준:** 홀수·짝수·음수·빈 입력 검사가 통과하고 두 앱이 기대값을 출력하며 `ctest --output-on-failure`가 모두 성공해야 합니다.

#### TODO 계약, 기본 빌드가 되는 이유, 권장 순서

| 작업 | 입력과 출력 | 수정되는 상태 | 완료를 확인하는 기준 |
|---|---|---|---|
| report target 활성화 | `src/report.cpp`와 `statistics` 라이브러리로 실행 파일 생성 | CMake 빌드 그래프에 `statistics_report` 소비자가 추가됨 | configure 후 파일이 만들어지고 `보고서 평균/중앙값: 5.00 / 5.00` 출력 |
| `mean` 구현 | 비어 있지 않은 `const vector<double>&`를 받아 평균 `double` 반환 | 호출자의 벡터는 바뀌지 않음; `<numeric>` 기능을 사용할 수 있게 include도 추가 | 기본 앱 입력과 report 입력 모두 평균 `5.0`; 빈 입력은 기존 검사에서 예외 |
| `median` 구현 | 벡터를 **값으로 복사** 해 받아 중앙값 `double` 반환 | 함수 안의 복사본만 정렬되고 호출자의 원래 순서는 유지 | `{8,3,5,4}`는 `4.5`, `{2,4,6,8}`은 `5.0`; 홀수는 정렬 후 가운데 한 값, 짝수는 가운데 두 값의 평균 |
| 테스트 활성화 | `summarize(samples)`의 평균·중앙값을 기대값과 비교 | 대표·짝수·홀수·음수·빈 입력·원본 보존 검사가 한 번에 실행됨 | 빌드한 뒤 `ctest --output-on-failure`에서 `statistics.unit` 통과 |

starter가 엄격한 `-Werror` 설정에서도 기본 컴파일되는 이유는 `mean`이 올바른 반환 형식의 임시값 `0.0`을, `median`이 빈 입력 검사 뒤 존재하는 첫 값을 반환하기 때문입니다. `summarize`의 count/min/max 구현은 이미 유효하고, 평균·중앙값의 전체 계약 검사는 스위치 뒤에 있어 starter의 기본 CTest도 통과합니다. report target은 선언되지 않아 아예 빌드 대상이 아닙니다. 즉 **빌드 성공과 현재 활성 테스트 통과는 TODO 정답을 보장하지 않습니다.**

권장 순서는 다음과 같습니다. (1) 헤더와 두 앱을 읽어 함수 계약과 기대 입력을 확인하고, (2) 필요한 표준 헤더를 추가해 평균을 구현하고, (3) 복사본을 이용해 중앙값을 구현합니다. (4) `statistics_app`을 빌드·실행해 수치를 확인한 뒤, (5) `full_contract_enabled`를 `true`로 바꾸어 전체 검사를 활성화한 뒤 다시 build → CTest를 수행합니다. (6) 마지막으로 report target을 활성화하고 **configure를 다시 실행** 한 뒤 전체 빌드, 두 앱 실행, CTest를 모두 확인하세요. 흐름은 `CMakeLists 저장 → configure가 그래프 생성 → build가 라이브러리와 소비자를 컴파일·링크 → 앱 직접 실행 → CTest가 등록된 테스트 실행`입니다.

- **변경·부작용:** 명세에서 지목한 지역 변수와 소유 상태만 변경합니다. 실행은 stdout을 출력하고 컴파일러가 build artifact를 만들지만 별도 데이터 파일이나 GUI 상태는 변경하지 않습니다.
- **오류·경계:** 명시된 값·수명·소유권 경계를 지키고, 컴파일 오류·예외·assert/CTest 실패는 숨기지 않습니다. 실패한 build 또는 run은 0이 아닌 종료 코드로 드러나야 합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
exercise_dir, _ = project_paths('exercise')
exercise_files = [
    exercise_dir / 'CMakeLists.txt',
    exercise_dir / 'src' / 'statistics.cpp',
    exercise_dir / 'tests' / 'statistics_tests.cpp',
]
for path in exercise_files:
    text = path.read_text(encoding='utf-8')
    todo_comments = sum(
        line.lstrip().startswith(('# TODO', '// TODO'))
        for line in text.splitlines()
    )
    print(path)
    print('  TODO 안내 주석:', todo_comments)

> **구분: 수업 후 복습·자습**

### 빌드·실행·테스트

저장한 뒤 아래 셀을 반복 실행하세요. 성공 기준은 다음과 같습니다.

- 기본 앱: 평균 `5.00`, 중앙값 `4.50`
- 추가 target `statistics_report`: `보고서 평균/중앙값: 5.00 / 5.00`
- CTest: `statistics.unit` 통과

`statistics_report`가 없다는 안내가 나오면 CMakeLists의 세 줄이 아직 주석인지 확인하세요. 전체 계약 스위치를 활성화하지 않으면 starter 기본 테스트만 통과할 수 있으므로 두 앱 출력과 활성화한 검사를 모두 확인합니다.
학습자 검증은 CTest의 전체 계약 완료 출력과 별도 경계 검사를 함께 확인합니다. `full_contract_enabled{true}`와 `full_contract_enabled = true`는 모두 유효한 초기화입니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
exercise_build = build('exercise')


In [ ]:
# 구분: 수업 후 복습·자습 셀
app_result = run([exercise_build / "statistics_app"])
expected_app = (
    "개수: 4\n"
    "최솟값/최댓값: 3.00 / 8.00\n"
    "평균: 5.00\n"
    "중앙값: 4.50\n"
)
assert app_result.stdout == expected_app, "statistics_app 출력을 확인하세요."

report_app = exercise_build / "statistics_report"
if not report_app.is_file():
    raise AssertionError("statistics_report target을 CMakeLists.txt에 추가하세요.")
report_result = run([report_app])
assert report_result.stdout == "보고서 평균/중앙값: 5.00 / 5.00\n"

test_result = run(["ctest", "--verbose", "-R", "^statistics[.]unit$"], cwd=exercise_build)
assert test_result.returncode == 0
# 기본 검사만 실행되거나 CTest 등록이 빠진 경우 완료로 처리하지 않습니다.
# true 초기화의 표기 방식 대신 전체 검사 끝에서만 나오는 실행 출력을 봅니다.
assert "모든 통계 계약을 통과했습니다." in test_result.stdout, (
    "statistics.unit를 등록하고 full_contract_enabled를 true로 활성화한 뒤 다시 빌드하세요."
)
hidden_contract_binary = exercise_build / "hidden_statistics_contract"
run([
    "g++", "-std=c++17", "-Wall", "-Wextra", "-Wpedantic", "-O0", "-g",
    "-I", exercise_dir / "include",
    exercise_dir / "src" / "statistics.cpp",
    LESSON_DIR / "tests" / "hidden_statistics_contract.cpp",
    "-o", hidden_contract_binary,
])
hidden_result = run([hidden_contract_binary])
assert hidden_result.stdout == "hidden statistics boundary contract PASS\n"
print("짝수·홀수·음수·빈 입력·원본 보존과 두 앱의 정확한 출력을 확인했습니다.")
print("learner_verification=PASS")

> **구분: 수업 후 복습·자습**

<details>
<summary><strong>참고 해답 빌드·실행</strong></summary>

직접 target 추가·구현·테스트를 먼저 통과시키세요. 아래 숨김 셀은 별도 `solution/` 프로젝트를 구성하므로 `exercise/` 파일을 덮어쓰지 않습니다. 해답 CMake에는 `statistics_report` target이 활성화되어 있고, 테스트는 짝수·홀수·음수·빈 입력과 원본 보존까지 검사합니다.

</details>

In [ ]:
# 구분: 수업 후 복습·자습 셀
# solution 프로젝트를 별도 빌드 디렉터리에 configure하고 전체 target을 컴파일합니다.
solution_build = build("solution")
solution_app_result = run([solution_build / "statistics_app"])
solution_report_result = run([solution_build / "statistics_report"])
# 짝수·홀수·음수·빈 입력·원본 보존 계약을 등록한 CTest를 실행합니다.
solution_result = run(["ctest", "--output-on-failure"], cwd=solution_build)

### 정답 해설 — 입력부터 결과까지

**target 연결을 읽기:** 헤더는 함수 약속, 구현 파일은 함수 몸체, CMake target은 어떤 파일을 함께 빌드할지 나타냅니다. 라이브러리를 만든 뒤 앱·테스트에서 해당 target을 연결해야 선언만 있고 정의가 없는 링크 오류를 피할 수 있습니다.

configure→build→CTest는 각각 규칙 준비→실행물 생성→기능 확인입니다. solution 디렉터리의 별도 빌드 성공은 exercise 프로젝트가 완성됐다는 뜻이 아닙니다. 같은 함수 이름이라도 서명·namespace·헤더 경로가 일치하는지 확인하세요.

**확인 질문:** 컴파일은 통과하고 undefined reference가 나오면 문법보다 먼저 어떤 target 연결을 확인하나요? 이 절은 자습입니다.

**적용·재검증:** 자신의 답안을 저장한 뒤 지정된 exercise 파일/작성 셀에 반영하고 저장→exercise 빌드→학습자 검증 순서로 실행합니다. solution 빌드는 별도 참고 결과입니다. 오류는 첫 compiler/CTest 메시지부터 읽습니다. `.so`를 이미 로드했다면 재빌드 후 새 Python 프로세스 또는 커널 재시작과 환경 셀 재실행으로 다시 로드합니다. 결과는 기존 Data 경로에 저장합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](06_cmake/solution/src/statistics.cpp)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```cpp
#include "statistics/statistics.hpp"

#include <algorithm>
#include <numeric>
#include <stdexcept>

namespace training::statistics {
namespace {

void require_values(const std::vector<double>& values) {
    if (values.empty()) {
        throw std::invalid_argument("통계를 계산할 값이 필요합니다.");
    }
}

}  // namespace

double mean(const std::vector<double>& values) {
    require_values(values);
    const double total = std::accumulate(values.begin(), values.end(), 0.0);
    return total / static_cast<double>(values.size());
}

double median(std::vector<double> values) {
    require_values(values);
    std::sort(values.begin(), values.end());

    const std::size_t middle = values.size() / 2U;
    if (values.size() % 2U == 1U) {
        return values.at(middle);
    }
    return (values.at(middle - 1U) + values.at(middle)) / 2.0;
}

Summary summarize(const std::vector<double>& values) {
    require_values(values);
    const auto bounds = std::minmax_element(values.begin(), values.end());

    return Summary{
        values.size(),
        *bounds.first,
        *bounds.second,
        mean(values),
        median(values),
    };
}

}  // namespace training::statistics
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 후 복습·자습 셀
assert solution_app_result.stdout == (
    "개수: 4\n최솟값/최댓값: 3.00 / 8.00\n평균: 5.00\n중앙값: 4.50\n"
)
assert solution_report_result.stdout == "보고서 평균/중앙값: 5.00 / 5.00\n"
assert solution_result.returncode == 0
print("solution_verification=PASS")

> **구분: 수업 후 복습·자습**

## 마무리 퀴즈 (56–60분)

<details><summary>1. configure와 build의 차이는?</summary>Configure는 CMakeLists와 환경을 읽어 빌드 시스템을 생성하고, build는 그 시스템으로 컴파일과 링크를 수행합니다.</details>

<details><summary>2. `statistics_app`만 빌드해도 `statistics`가 먼저 만들어지는 이유는?</summary>`target_link_libraries(statistics_app PRIVATE statistics)`가 target 의존 관계를 만들기 때문입니다.</details>

<details><summary>3. 공개 헤더 경로가 `PUBLIC`인 이유는?</summary>라이브러리 자신뿐 아니라 링크하는 앱과 테스트도 그 헤더를 include해야 하기 때문입니다.</details>

<details><summary>4. 테스트 실행 파일을 빌드하면 CTest도 자동으로 실행될까요?</summary>아닙니다. 빌드와 테스트 실행은 별도 단계이며, 이 과정에서는 빌드 디렉터리를 작업 디렉터리로 삼아 `ctest --output-on-failure`를 실행합니다.</details>

<details><summary>5. 새 `statistics_report` 실행 파일을 추가하려면 어떤 세 관계를 선언하나요?</summary>`add_executable`로 소스를 target에 묶고, `target_link_libraries`로 statistics에 연결하며, 과정의 helper로 엄격 경고를 적용합니다.</details>

### 셀프 체크

- [ ] 소스 디렉터리와 빌드 디렉터리를 분리할 수 있다.
- [ ] `add_executable`과 `target_link_libraries`로 새 target을 직접 추가할 수 있다.
- [ ] 라이브러리·앱·테스트 target의 관계를 설명할 수 있다.
- [ ] `PUBLIC`, `PRIVATE`, `INTERFACE`를 구분할 수 있다.
- [ ] 실패한 CTest의 출력을 `--output-on-failure`로 확인할 수 있다.

> **구분: 수업 후 복습·자습**

## 한 줄 정리

**CMake에서는 명령 모음보다 target과 사용 요구사항의 그래프를 선언하고, configure → build → test 단계를 분리해 검증합니다.**